In [1]:
import pandas as pd
import numpy as np
import torch
from  torch.optim import AdamW, Adam, SGD, RMSprop
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments, get_linear_schedule_with_warmup
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import gc
from transformers import EarlyStoppingCallback
from utils.config import *

In [2]:
dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'

# q type classifer,
q_type_model_name= 'distilbert-base-uncased'
q_type_model_result= '.temp/model_results/q_types_model_lite_results'
q_type_model= '.temp/model/fine_tuned_question_classifier_model_lite'



# Question type classsifier

### Preprocessing

In [3]:
df= pd.read_csv(dataset_path)
df= df[["question", "question_type"]]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1644 entries, 0 to 1643
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1644 non-null   object
 1   question_type  1644 non-null   object
dtypes: object(2)
memory usage: 25.8+ KB


In [4]:
df["question_type"].unique()

array(['current_ctc', 'expected_ctc', 'personal_information', 'education',
       'working_experince', 'skills', 'availability', 'others'],
      dtype=object)

In [5]:
df.drop_duplicates(inplace= True)

In [6]:
# adding labels
label_mapping = {key: index for index, key in enumerate(QUESTION_TYPES)}
df['label'] = df['question_type'].map(label_mapping)

# droping unused column
df.drop('question_type', axis=1,  inplace= True)
df.head()

,question,label
0,What is your current CTC?,0
1,Can you share your current salary package?,0
2,What's your present compensation?,0
3,How much are you earning right now?,0
4,Could you tell me your current pay scale?,0


In [7]:
df= df.sample(frac=1).reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1634 entries, 0 to 1633
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  1634 non-null   object
 1   label     1634 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 25.7+ KB


In [8]:
df.head(5)

,question,label
0,Are there any handover responsibilities delayi...,6
1,What's your paycheck amount each month?,0
2,Subject focus of study?,3
3,Do you prefer direct or indirect communication...,7
4,How many disaster recovery plans have you deve...,5


### Retraing Preparations:

In [9]:
#convert to hugging face dataset
dataset= Dataset.from_pandas(df)

#Split the data into train and test sets (80-20 split)
dataset_split = dataset.train_test_split(test_size=0.2)

# Access train and test splits
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']

In [10]:
tokenizer = DistilBertTokenizerFast.from_pretrained(q_type_model_name)

In [11]:
def tokenize_function(examples):
    return tokenizer(examples['question'], padding= "max_length", truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set the format to PyTorch tensors
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/1307 [00:00<?, ? examples/s]

Map:   0%|          | 0/327 [00:00<?, ? examples/s]

In [12]:
# Mapping lebel and id
id2label = {v: k for k, v in label_mapping.items()}  # Map IDs to label names
label2id = {k: v for k, v in label_mapping.items()}  # Map label names to IDs

# Retraning

# using default optimizer

In [13]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_name,
        num_labels=8,
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


8


In [14]:
training_args = TrainingArguments(
    output_dir= q_type_model_result+"-default",           # Output directory
    eval_strategy="epoch",     # Evaluate after a specific number of steps
    save_strategy="epoch",           # Save the model after a specific number of steps
    learning_rate= 3e-5,
    num_train_epochs=50,             # Number of training epochs
    per_device_train_batch_size= 16,   # Batch size per device during training
    per_device_eval_batch_size= 16,    # Batch size per device during evaluation
    gradient_accumulation_steps=2,
    logging_dir='./logs',            # Directory for storing logs
    logging_steps=10,                # Log every 10 steps
    load_best_model_at_end=True,     # Required for EarlyStoppingCallback
    # use_cpu=True                     # Force CPU usage (but TrainingArguments doesn’t support this; see notes below)
)

In [15]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.144000,0.930738,0.792049
2,0.538700,0.470270,0.856269
3,0.276500,0.301106,0.911315
4,0.149300,0.282457,0.908257
5,0.083600,0.288605,0.908257
6,0.046600,0.272907,0.932722
7,0.014600,0.294320,0.926606
8,0.013700,0.319731,0.911315


TrainOutput(global_step=328, training_loss=0.3597910862597751, metrics={'train_runtime': 1660.182, 'train_samples_per_second': 39.363, 'train_steps_per_second': 1.235, 'total_flos': 1385227325865984.0, 'train_loss': 0.3597910862597751, 'epoch': 8.0})

- seems like epoch 6 will be best for prediction

### Model evaluation and Saving

In [16]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.2729066014289856,
 'eval_accuracy': 0.9327217125382263,
 'eval_runtime': 19.5864,
 'eval_samples_per_second': 16.695,
 'eval_steps_per_second': 1.072,
 'epoch': 8.0}

In [17]:
trainer.save_model(q_type_model+"-default")
tokenizer.save_pretrained(q_type_model+"-default")

('.temp/model/fine_tuned_question_classifier_model_lite-default/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/tokenizer.json')

# Using AdamW optimiser with linear scheduler with warmup

In [18]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_name,
        num_labels=8,
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


8


In [19]:
training_args = TrainingArguments(
    output_dir= q_type_model_result+"-AdamW",           # Output directory
    eval_strategy="epoch",     # Evaluate after a specific number of steps
    save_strategy="epoch",           # Save the model after a specific number of steps
    learning_rate= 3e-5,
    num_train_epochs=50,             # Number of training epochs
    per_device_train_batch_size= 16,   # Batch size per device during training
    per_device_eval_batch_size= 16,    # Batch size per device during evaluation
    gradient_accumulation_steps=2,
    logging_dir='./logs',            # Directory for storing logs
    logging_steps=10,                # Log every 10 steps
    load_best_model_at_end=True,     # Required for EarlyStoppingCallback
    # use_cpu=True                     # Force CPU usage (but TrainingArguments doesn’t support this; see notes below)
)

In [20]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = AdamW(model.parameters(), lr=5e-5, eps=1e-8)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [21]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,2.060300,2.040493,0.143731
2,1.696800,1.544302,0.706422
3,1.038000,0.859088,0.761468
4,0.562200,0.577279,0.816514
5,0.356300,0.326793,0.914373
6,0.137600,0.284729,0.923547
7,0.085600,0.324693,0.914373
8,0.031100,0.359436,0.914373


TrainOutput(global_step=328, training_loss=0.8196319133588453, metrics={'train_runtime': 1782.0172, 'train_samples_per_second': 36.672, 'train_steps_per_second': 1.15, 'total_flos': 1385227325865984.0, 'train_loss': 0.8196319133588453, 'epoch': 8.0})

- seems like epoch 6 will be best for prediction

In [22]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.28472915291786194,
 'eval_accuracy': 0.9235474006116208,
 'eval_runtime': 21.5251,
 'eval_samples_per_second': 15.192,
 'eval_steps_per_second': 0.976,
 'epoch': 8.0}

In [23]:
trainer.save_model(q_type_model+"-AdamW")
tokenizer.save_pretrained(q_type_model+"-AdamW")

('.temp/model/fine_tuned_question_classifier_model_lite-AdamW/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/tokenizer.json')

# Using Adam

In [24]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_name,
        num_labels=8,
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


8


In [25]:
training_args = TrainingArguments(
    output_dir= q_type_model_result+"-Adam",           # Output directory
    eval_strategy="epoch",     # Evaluate after a specific number of steps
    save_strategy="epoch",           # Save the model after a specific number of steps
    learning_rate= 3e-5,
    num_train_epochs=50,             # Number of training epochs
    per_device_train_batch_size= 16,   # Batch size per device during training
    per_device_eval_batch_size= 16,    # Batch size per device during evaluation
    gradient_accumulation_steps=2,
    logging_dir='./logs',            # Directory for storing logs
    logging_steps=10,                # Log every 10 steps
    load_best_model_at_end=True,     # Required for EarlyStoppingCallback
    # use_cpu=True                     # Force CPU usage (but TrainingArguments doesn’t support this; see notes below)
)

In [26]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = Adam(model.parameters(), lr=3e-5)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [27]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,2.074000,2.066410,0.100917
2,1.913300,1.828976,0.483180
3,1.431900,1.252747,0.752294
4,0.880200,0.787276,0.770642
5,0.573200,0.521031,0.874618
6,0.321700,0.361247,0.899083
7,0.182200,0.354438,0.889908
8,0.131800,0.273498,0.914373
9,0.075900,0.316397,0.905199
10,0.053700,0.365079,0.892966


TrainOutput(global_step=410, training_loss=0.8176018569527603, metrics={'train_runtime': 2242.3514, 'train_samples_per_second': 29.144, 'train_steps_per_second': 0.914, 'total_flos': 1731534157332480.0, 'train_loss': 0.8176018569527603, 'epoch': 10.0})

- seems like epoch 8 will be best for prediction

In [28]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.2734982967376709,
 'eval_accuracy': 0.9143730886850153,
 'eval_runtime': 20.0904,
 'eval_samples_per_second': 16.276,
 'eval_steps_per_second': 1.045,
 'epoch': 10.0}

In [29]:
trainer.save_model(q_type_model+"-Adam")
tokenizer.save_pretrained(q_type_model+"-Adam")

('.temp/model/fine_tuned_question_classifier_model_lite-Adam/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/tokenizer.json')

# Using SGD

In [30]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_name,
        num_labels=8,
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


8


In [31]:
training_args = TrainingArguments(
    output_dir= q_type_model_result+"-SGD",           # Output directory
    eval_strategy="epoch",     # Evaluate after a specific number of steps
    save_strategy="epoch",           # Save the model after a specific number of steps
    learning_rate= 3e-5,
    num_train_epochs=50,             # Number of training epochs
    per_device_train_batch_size= 16,   # Batch size per device during training
    per_device_eval_batch_size= 16,    # Batch size per device during evaluation
    gradient_accumulation_steps=2,
    logging_dir='./logs',            # Directory for storing logs
    logging_steps=10,                # Log every 10 steps
    load_best_model_at_end=True,     # Required for EarlyStoppingCallback
    # use_cpu=True                     # Force CPU usage (but TrainingArguments doesn’t support this; see notes below)
)

In [32]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [33]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,2.067300,2.062855,0.140673
2,1.965200,1.886178,0.443425
3,1.485000,1.277823,0.688073
4,0.864000,0.775527,0.804281
5,0.544300,0.515647,0.828746
6,0.375300,0.422484,0.862385
7,0.251800,0.422660,0.837920
8,0.180900,0.282476,0.911315
9,0.137300,0.256299,0.905199
10,0.154300,0.233766,0.920489


TrainOutput(global_step=492, training_loss=0.7248466438575973, metrics={'train_runtime': 2707.1921, 'train_samples_per_second': 24.139, 'train_steps_per_second': 0.757, 'total_flos': 2077840988798976.0, 'train_loss': 0.7248466438575973, 'epoch': 12.0})

- seems like epoch 10 will be best for prediction

In [34]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.23376613855361938,
 'eval_accuracy': 0.9204892966360856,
 'eval_runtime': 20.292,
 'eval_samples_per_second': 16.115,
 'eval_steps_per_second': 1.035,
 'epoch': 12.0}

In [35]:
trainer.save_model(q_type_model+"-SGD")
tokenizer.save_pretrained(q_type_model+"-SGD")

('.temp/model/fine_tuned_question_classifier_model_lite-SGD/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/tokenizer.json')

# Using RMsprop

In [36]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_name,
        num_labels=8,
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


8


In [37]:
training_args = TrainingArguments(
    output_dir= q_type_model_result+"-RMSprop",           # Output directory
    eval_strategy="epoch",     # Evaluate after a specific number of steps
    save_strategy="epoch",           # Save the model after a specific number of steps
    learning_rate= 3e-5,
    num_train_epochs=50,             # Number of training epochs
    per_device_train_batch_size= 16,   # Batch size per device during training
    per_device_eval_batch_size= 16,    # Batch size per device during evaluation
    gradient_accumulation_steps=2,
    logging_dir='./logs',            # Directory for storing logs
    logging_steps=10,                # Log every 10 steps
    load_best_model_at_end=True,     # Required for EarlyStoppingCallback
    # use_cpu=True                     # Force CPU usage (but TrainingArguments doesn’t support this; see notes below)
)

In [38]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = RMSprop(model.parameters(), lr=0.01, alpha=0.99)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [39]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,2.147300,2.107378,0.116208
2,2.125300,2.118784,0.149847
3,2.110500,2.094643,0.125382
4,2.082700,2.081744,0.131498
5,2.229800,2.080696,0.131498
6,2.074200,2.081956,0.131498
7,2.076400,2.083181,0.131498


TrainOutput(global_step=287, training_loss=2.1209855262410766, metrics={'train_runtime': 1575.394, 'train_samples_per_second': 41.482, 'train_steps_per_second': 1.301, 'total_flos': 1212073910132736.0, 'train_loss': 2.1209855262410766, 'epoch': 7.0})

In [40]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 2.080695629119873,
 'eval_accuracy': 0.13149847094801223,
 'eval_runtime': 20.3499,
 'eval_samples_per_second': 16.069,
 'eval_steps_per_second': 1.032,
 'epoch': 7.0}

In [41]:
trainer.save_model(q_type_model+"-RMSprop")
tokenizer.save_pretrained(q_type_model+"-RMSprop")

('.temp/model/fine_tuned_question_classifier_model_lite-RMSprop/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-RMSprop/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-RMSprop/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-RMSprop/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-RMSprop/tokenizer.json')